# Step 1 Phase A-8a：ℓ≤16 B⁺ 特徴スタック（S2 hybrid selection surrogate 用）の構築・正式検証・atomic 保存 **v1.1.2**
2026-09-09。v1.1.1 監査（CODE DESIGN APPROVED・official GO）の軽微推奨：float32 適格性を **event-output 用と axis-output 用に分離**（後者は軸不一致例に same plane または plane-folded 角距離 ≤ 隣接許容を要求），`t1_engine.py` を live SHA gate に追加，相対差分母の tiny guard，軸不一致件数・plane-folded 分離の要約。
v1.1.1：v1.1 監査（2026-09-09 #2）：float32 Event B gate が f32 選択軸を評価していなかった欠陥を修正（T₁/T₂ を ref/f64/f32 の3軸で保存）／runner-up は winner のみ除外した **raw margin** に戻し，
対蹠軸は plane-equivalent 診断（antipode score gap・R 行同一・same_plane）として別記録／near-tie で軸が変わる場合は selection score・T₁・T₂・Event B の一致を要求／A11 共分散の実基底 binding（C4 == Re(M4·Csym·M4ᴴ)）を gate。
v1.0 監査（2026-09-09）の 5 BLOCKER に対応：**高ℓ入力（FL/CL/W4/W16）を A5 input manifest v2 に hard binding**／A11 共分散の一意選択と A11 freeze の status 検証／
**float32 の正式適格性 gate**（near-tie 分離・`float32_eligible_for_rules` 出力）／観測 10 マップの期待 SHA と exact inventory／A11 方式の smoke→official 証拠鎖／manifest↔provenance 相互 binding／near-tie 電池の完成。
2026-09-08。A8 は監査（2026-09-08）の指示で **A8a（構築＋検証）** と **A8b（隔離 subprocess ベンチマーク）** に分割。本 notebook は A8a。

- 凍結資産は **A5 freeze manifest（SHA 固定）と A5 provenance から読む**（mask/R/valid/cnt・B-stack file/array SHA の手入力二重管理を回避）。
- 基底拡張電池（凍結順序が接頭辞・M 行列の部分行列一致）→ Y16 → F16（3072×40,755）。**全 3072 軸**で ℓ≤4 ブロックが A5 B⁺ と一致することを gate。
- 正式検証電池：観測マップ（CMBanom 8 本＋PR4 があれば 2 本）・fresh 等方 1000・**hybrid サンプル**（A11 凍結 E7 共分散の ℓ2–4 ＋ 等方 ℓ5–16）200，
  float64 特徴経路と exact float32 特徴経路について，全軸 score 残差・選択 T₁ 残差・argmin 一致・角距離・second-best gap・near-tie・
  選択軸の ℓ2–4 T₁/T₂・Event B indicator 不一致を保存。dtype 一致率は **top-two margin 別**に集計。
- atomic 保存（`.npy`＋JSON manifest・fsync・`os.replace`）→ fresh reopen（mmap）で shape/dtype/SHA/axis id/basis/packing を再検査 → `FEATURESTACK_VALID`。
- モード：`A8_MODE='smoke'`（縮小・環境 lock 生成）／`'official'`（Colab 必須・lock 一致・notebook 同一性）。

In [ ]:
# ---- 1. モード・環境・ソース同一性（A11 方式）----
import os, sys, json, hashlib, subprocess, time, datetime, platform, importlib, inspect, glob, shutil
from pathlib import Path
from importlib.metadata import version as pkg_version, PackageNotFoundError
A8_MODE = globals().get('A8_MODE', os.environ.get('A8_MODE', 'official')); assert A8_MODE in ('smoke', 'official'), A8_MODE
IN_COLAB = os.path.isdir('/content')
WORK = '/content' if IN_COLAB else os.environ.get('A8_WORK', os.path.join(os.getcwd(), 'a8_work'))
BASE = '/content/drive/MyDrive/mirror_topology' if IN_COLAB else os.environ.get('A8_BASE', os.path.join(WORK, 'base'))
if IN_COLAB:
    if not os.path.isdir('/content/drive/MyDrive'):
        from google.colab import drive; drive.mount('/content/drive')
    assert os.path.isdir('/content/drive/MyDrive')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'healpy==1.20.0', 'psutil', 'threadpoolctl'], check=True)
os.makedirs(WORK, exist_ok=True); OUT = os.path.join(BASE, 'runs_step1_phaseA', f'a8a_v1.1.2_{A8_MODE}'); os.makedirs(OUT, exist_ok=True)
LOCK_PATH = os.path.join(BASE, 'runs_step1_phaseA', 'a8_env_lock.json'); NB_BASENAME = 'MirrorTopology_Step1_A8a_featurestack_v1.1.2.ipynb'
GATES = {}; DIAG = {}; GIT_LOG = []
def sha256_file(p, block=8 << 20):
    h = hashlib.sha256()
    with open(p, 'rb') as fh:
        for b in iter(lambda: fh.read(block), b''): h.update(b)
    return h.hexdigest()
def run(c, **kw):
    r = subprocess.run(c, capture_output=True, text=True, check=True, **kw); GIT_LOG.append(dict(cmd=c, stderr=r.stderr.strip()[:200])); return r.stdout.strip()
_env = dict(os.environ, GIT_TERMINAL_PROMPT='0')
def pin_repo(url, path, commit, canonical):
    if not os.path.isdir(os.path.join(path, '.git')): run(['git', 'clone', url, path], env=_env)
    if run(['git', '-C', path, 'rev-parse', '--is-shallow-repository']) == 'true': run(['git', '-C', path, 'fetch', '-q', '--unshallow'], env=_env)
    run(['git', '-C', path, 'fetch', '-q', 'origin', commit], env=_env); run(['git', '-C', path, 'reset', '-q', '--hard', commit]); run(['git', '-C', path, 'clean', '-fdx', '-q'])
    return dict(head=run(['git', '-C', path, 'rev-parse', 'HEAD']), origin=run(['git', '-C', path, 'remote', 'get-url', 'origin']), clean=(run(['git', '-C', path, 'status', '--porcelain']) == ''), canonical=canonical)
PINS = dict(pem=('https://github.com/tsujikeita/plane-excised-mirror.git', 'd36e7567e8a7869c0d7b84955b4139ab0e782af0', 'https://github.com/tsujikeita/plane-excised-mirror'),
            cmbanom=('https://github.com/LauraHerold/CMBanom.git', 'aaf8137427d54ce4c77e59734391aca491a4a8db', 'https://github.com/LauraHerold/CMBanom'),
            mt=('https://github.com/tsujikeita/mirror-topology.git', '5ad38e12c4d769f3ddbfb95b2cff18d959bfc0e7', 'https://github.com/tsujikeita/mirror-topology'))
PEM, CA, MT = (os.path.join(WORK, 'pem_a8'), os.path.join(WORK, 'CMBanom'), os.path.join(WORK, 'mt_a8'))
REPOS = {}
for name, path in [('pem', PEM), ('cmbanom', CA), ('mt', MT)]:
    url, commit, canon = PINS[name]; r = pin_repo(url, path, commit, canon); REPOS[name] = dict(commit=commit, **r)
    GATES[f'G_{name}_commit'] = (r['head'] == commit); GATES[f'G_{name}_origin'] = (r['origin'].rstrip('/').removesuffix('.git') == canon); GATES[f'G_{name}_clean_preimport'] = r['clean']
assert all(GATES.values()), GATES
# ---- A5 freeze manifest binding (single source of truth for frozen hashes) ----
A5 = os.path.join(MT, 'results', 'step1_phaseA', 'A5_freeze')
GATES['G_a5_manifest_sha'] = (sha256_file(os.path.join(A5, 'A5_freeze_manifest.json')) == '79419d00e36a4ec1a7abb77a0e0a5ccf07e0170c3df10b48052b637492ba9ff2')
a5m = json.load(open(os.path.join(A5, 'A5_freeze_manifest.json'))); a5p = json.load(open(os.path.join(A5, 'a5_provenance.json'))); a5b = json.load(open(os.path.join(A5, 's1_Bstack_l2_4_provenance.json')))
GATES['G_a5_provenance_in_manifest'] = (sha256_file(os.path.join(A5, 'a5_provenance.json')) == a5m['outputs']['a5_provenance.json'] and sha256_file(os.path.join(A5, 's1_Bstack_l2_4_provenance.json')) == a5m['outputs']['s1_Bstack_l2_4_provenance.json'])
FROZEN = dict(processed_mask=a5p['input_bitlevel_sha']['processed_mask'], R=a5p['input_bitlevel_sha']['reflection_table'], valid=a5p['input_bitlevel_sha']['valid_table'], cnt=a5p['input_bitlevel_sha']['cnt'],
              bstack_file=a5b['npz_sha256'], bstack_array=a5b['bstack_array_sha256'], pem_src=a5p['pem']['src_sha'])
BST = os.path.join(A5, 's1_Bstack_l2_4_N16_common_v1.npz'); GATES['G_a5_bstack_file_sha'] = (sha256_file(BST) == FROZEN['bstack_file'])
EXPECTED_SHA = dict(bridge='45107d1608d50816712f1aa452d9fa39af4adc9ec035fbe9279b264760d65872', run='03c80f2136a8ff7ffb1077749895811ef95dd9d779c7996891d89e75545ff8db',
                    t1_engine='87bf8424073af021264b12fe312ab5255b71008bdd5fe874d164d48daf034dc8')
GATES['G_pem_src_sha'] = all(sha256_file(os.path.join(PEM, f)) == v for f, v in FROZEN['pem_src'].items())
GATES['G_mt_src_sha'] = (sha256_file(os.path.join(MT, 't2b2_bridge.py')) == EXPECTED_SHA['bridge'] and sha256_file(os.path.join(MT, 't2b2_run.py')) == EXPECTED_SHA['run'] and sha256_file(os.path.join(MT, 't1_engine.py')) == EXPECTED_SHA['t1_engine'])
assert all(GATES.values()), GATES
# ---- environment lock ----
PKGS = ['numpy', 'scipy', 'healpy', 'pandas', 'psutil', 'threadpoolctl']
def pkgver(n):
    try: return pkg_version(n)
    except PackageNotFoundError: return 'MISSING'
VERS = {n: pkgver(n) for n in PKGS}; PYV = sys.version.split()[0]
for name in list(sys.modules):
    if name in {'t1_engine', 't2b2_bridge', 't2b2_run', 'phase2_core', 'plane_mirror'}: del sys.modules[name]
importlib.invalidate_caches(); os.chdir(WORK); sys.path.insert(0, os.path.join(PEM, 'src')); sys.path.insert(0, MT)
import numpy as np, pandas as pd, healpy as hp, scipy, psutil
import phase2_core as p2, plane_mirror as pm, t2b2_bridge as br, t2b2_run as tr, t1_engine as t1
LIVE = dict(numpy=np.__version__, scipy=scipy.__version__, healpy=hp.__version__, pandas=pd.__version__, psutil=psutil.__version__, threadpoolctl=pkgver('threadpoolctl'))
GATES['G_live_dependency_versions'] = ({k: str(v) for k, v in VERS.items()} == {k: str(v) for k, v in LIVE.items()}); assert GATES['G_live_dependency_versions'], (VERS, LIVE)
GATES['G_live_import_paths'] = all(Path(m.__file__).resolve() == Path(p).resolve() for m, p in [(p2, os.path.join(PEM, 'src', 'phase2_core.py')), (pm, os.path.join(PEM, 'src', 'plane_mirror.py')), (br, os.path.join(MT, 't2b2_bridge.py')), (tr, os.path.join(MT, 't2b2_run.py')), (t1, os.path.join(MT, 't1_engine.py'))])
GATES['G_live_module_sha'] = (sha256_file(br.__file__) == EXPECTED_SHA['bridge'] and sha256_file(tr.__file__) == EXPECTED_SHA['run'] and sha256_file(t1.__file__) == EXPECTED_SHA['t1_engine'] and all(sha256_file(m.__file__) == FROZEN['pem_src'][k] for m, k in [(pm, 'src/plane_mirror.py'), (p2, 'src/phase2_core.py')]))
assert GATES['G_live_import_paths'] and GATES['G_live_module_sha']
PLATFORM = dict(machine=platform.machine(), platform=platform.platform(), cpu_count=os.cpu_count(), ram_GB=round(psutil.virtual_memory().total / 1e9, 2))
BLAS = str({k: v.get('name') for k, v in np.show_config(mode='dicts').get('Build Dependencies', {}).items() if k in ('blas', 'lapack')})
LOCK_CANDIDATE = dict(schema='a8_env_lock_v1', python=PYV, versions=LIVE, platform=dict(machine=PLATFORM['machine'], platform=PLATFORM['platform']), blas_lapack=BLAS, smoke_status='PENDING')
if A8_MODE == 'official':
    GATES['G_colab_runtime'] = IN_COLAB
    assert os.path.exists(LOCK_PATH), 'official requires the smoke environment lock'
    lock = json.load(open(LOCK_PATH)); LOCK_SHA = sha256_file(LOCK_PATH)
    GATES['G_env_lock'] = (lock.get('schema') == 'a8_env_lock_v1' and lock['python'] == PYV and lock['versions'] == LIVE and lock['blas_lapack'] == BLAS and lock['platform']['machine'] == PLATFORM['machine'] and lock.get('smoke_status') == 'SMOKE_PASS')
    assert GATES['G_env_lock'] and GATES['G_colab_runtime'], (GATES, lock)
else: lock = None; LOCK_SHA = None; GATES['G_env_lock_candidate_complete'] = True
ENV_FINGERPRINT = hashlib.sha256(json.dumps(dict(python=PYV, versions=LIVE, blas=BLAS, machine=PLATFORM['machine']), sort_keys=True).encode()).hexdigest()
run(['git', '-C', MT, 'fetch', '-q', 'origin', 'main'], env=_env); MAIN_HEAD = run(['git', '-C', MT, 'rev-parse', 'origin/main'])
try: NB_HEAD_SHA = tr.source_only_sha(subprocess.run(['git', '-C', MT, 'show', f'origin/main:{NB_BASENAME}'], capture_output=True, check=True).stdout)
except subprocess.CalledProcessError: NB_HEAD_SHA = None
NB_LIVE_SHA = tr.live_notebook_source_sha(); DIAG['notebook'] = dict(live=NB_LIVE_SHA, origin_main_head=MAIN_HEAD, head_copy=NB_HEAD_SHA)
if A8_MODE == 'official':
    GATES['G_notebook_live_source'] = bool(isinstance(NB_LIVE_SHA, str) and NB_HEAD_SHA is not None and NB_LIVE_SHA == NB_HEAD_SHA); assert GATES['G_notebook_live_source'], DIAG['notebook']
    chain = dict(ok=False)
    try:
        spath = lock.get('smoke_provenance_path'); path_ok = isinstance(spath, str) and os.path.isfile(spath); sha_ok = bool(path_ok and sha256_file(spath) == lock.get('smoke_provenance_sha256')); sp = json.load(open(spath)) if sha_ok else {}
        req = sp.get('required_gates'); status_ok = (sp.get('status') == 'SMOKE_PASS'); gates_ok = (isinstance(req, list) and len(req) > 0 and sp.get('gate_inventory_exact') is True and all(sp.get('gates', {}).get(k) is True for k in req))
        source_ok = (lock.get('canonical_notebook_head_sha') == NB_HEAD_SHA and sp.get('notebook_identity', {}).get('head_copy') == NB_HEAD_SHA); env_ok = (sp.get('environment', {}).get('fingerprint') == ENV_FINGERPRINT and lock['platform']['platform'] == PLATFORM['platform'])
        chain.update(path=spath, path_ok=path_ok, sha_ok=sha_ok, status_ok=status_ok, gates_ok=gates_ok, source_ok=source_ok, environment_ok=env_ok); chain['ok'] = bool(path_ok and sha_ok and status_ok and gates_ok and source_ok and env_ok)
    except Exception as e: chain['error'] = f'{type(e).__name__}: {e}'
    DIAG['smoke_provenance_chain'] = chain; GATES['G_smoke_provenance_chain'] = bool(chain['ok']); assert GATES['G_smoke_provenance_chain'], chain
else: GATES['G_notebook_head_available'] = (NB_HEAD_SHA is not None)
print(f'mode={A8_MODE} / repos pinned / env fingerprint {ENV_FINGERPRINT[:12]} / {PLATFORM} / {LIVE}')

In [ ]:
# ---- 2. 凍結パイプライン objects と基底拡張電池 ----
def sha256_array(a, block=8 << 20):
    a = np.ascontiguousarray(a); mv = memoryview(a).cast('B'); h = hashlib.sha256()
    for i in range(0, len(mv), block): h.update(mv[i:i + block])
    return h.hexdigest()
LMAX_SRC, NSIDE = 128, 16
ms = pm.with_mask(p2.MirrorStat(NSIDE, p2.make_mask(NSIDE, 'full')), p2.make_mask(NSIDE, 'common')); NPIX = ms.R.shape[0]
GATES['G_pipeline_objects'] = bool(NPIX == 3072 and ms.R.shape == (3072, 3072) and ms.valid.shape == (3072, 3072) and ms.cnt.shape == (3072,) and (ms.cnt > 0).all())
GATES['G_mask_R_valid_cnt_sha'] = (sha256_array(ms.mask.astype(np.uint8)) == FROZEN['processed_mask'] and sha256_array(ms.R) == FROZEN['R'] and sha256_array(ms.valid) == FROZEN['valid'] and sha256_array(ms.cnt) == FROZEN['cnt'])
assert GATES['G_pipeline_objects'] and GATES['G_mask_R_valid_cnt_sha']
FL = p2.transfer(NSIDE, 'planck', LMAX_SRC); ELL = np.arange(LMAX_SRC + 1); CL = p2.load_fid_cl()
W4 = ((ELL >= 2) & (ELL <= 4)).astype(float) * FL; W16 = ((ELL >= 2) & (ELL <= 16)).astype(float) * FL
INPUT_SHA = dict(FL=sha256_array(FL), W4=sha256_array(W4), W16=sha256_array(W16), CL=sha256_array(CL), pixwin_source_diagnostic=('healpy_download' if not os.path.exists(os.path.join(p2.PIXWIN_CACHE, 'pixel_window_n0016.fits')) else 'local_cache'))
A5IM = os.path.join(A5, 'a5_input_manifest_v2.json'); GATES['G_a5_input_manifest_sha'] = (sha256_file(A5IM) == 'ae2dd6feb79f6016664df44e6bc35630e60f6df623512dcfbe6761389752e6a6'); assert GATES['G_a5_input_manifest_sha']
a5im = json.load(open(A5IM))
def scaled_diff(cur, ref):
    ref = np.asarray(ref, float); cur = np.asarray(cur, float); assert ref.shape == cur.shape, (ref.shape, cur.shape); return float(np.abs(cur - ref).max() / max(np.abs(ref).max(), np.finfo(float).tiny))
INPUT_BIND = dict(FL=scaled_diff(FL, a5im['float_ref']['fl']), CL=scaled_diff(CL, a5im['float_ref']['cl_array']), W4=scaled_diff(W4, a5im['float_ref']['selwin_l2_4']), W16=scaled_diff(W16, a5im['float_ref']['selwin_l16']))
for k, v in INPUT_BIND.items(): GATES[f'G_{k}_matches_A5'] = (v <= a5im['float_rtol'])
DIAG['input_binding_scaled_diff'] = INPUT_BIND; assert all(GATES[f'G_{k}_matches_A5'] for k in INPUT_BIND), INPUT_BIND
LS16 = list(range(2, 17)); M16, LM16, RB16 = br.M_matrix(LS16); M4, LM4, RB4 = br.M_matrix()
cols4 = [LM16.index(t) for t in LM4]
GATES['G_basis_extension'] = bool(RB16[:21] == RB4 and len(RB16) == 285 and len(LM16) == 285 and np.abs(M16[:21][:, cols4] - M4).max() == 0 and np.abs(M16[:21]).sum() == np.abs(M16[:21][:, cols4]).sum())
assert GATES['G_basis_extension']
IDX16 = {t: k for k, t in enumerate(LM16)}
def x_to_map(x, M, LM, W):
    a = x @ np.conj(M); alm = np.zeros(hp.Alm.getsize(LMAX_SRC), complex)
    for (l, m) in LM:
        if m >= 0: alm[hp.Alm.getidx(LMAX_SRC, l, m)] = a[IDX16[(l, m)] if len(LM) == 285 else LM.index((l, m))]
    mb = hp.alm2map(hp.almxfl(alm, W), NSIDE)
    return np.where(ms.mask, np.asarray(hp.remove_dipole(hp.ma(np.where(ms.mask, mb, hp.UNSEEN)))), 0.0)
E285 = np.eye(285); E21 = np.eye(21); t0 = time.time()
Y16 = np.column_stack([x_to_map(E285[i], M16, LM16, W16) for i in range(285)]); Y4 = np.column_stack([x_to_map(E21[i], M4, LM4, W4) for i in range(21)])
GATES['G_Y16_prefix_equals_Y4'] = bool(np.abs(Y16[:, :21] - Y4).max() < 1e-12); assert GATES['G_Y16_prefix_equals_Y4']
print(f'Y16 {Y16.shape} built {time.time()-t0:.0f}s; prefix gate OK')

In [ ]:
# ---- 3. F16 構築（全 3072 軸 ℓ≤4 ブロック gate）----
with np.load(BST) as zB: Bp4 = zB['Bp_stack'].copy(); Bm4 = zB['Bm_stack'].copy()
GATES['G_a5_bstack_array_sha'] = (hashlib.sha256(np.ascontiguousarray(Bp4).tobytes() + np.ascontiguousarray(Bm4).tobytes()).hexdigest() == FROZEN['bstack_array']); assert GATES['G_a5_bstack_array_sha']
D = 285; iu = np.triu_indices(D); NF = len(iu[0]); assert NF == 40755; off = (iu[0] != iu[1])
def fvec(B): v = B[iu].copy(); v[off] *= 2.0; return v
def packed_features_into(x, out):        # out[:, k] = x_i x_j (i<=j), no CH x D x D temporary
    k = 0; d = x.shape[1]
    for i in range(d):
        w = d - i; np.multiply(x[:, i:i + 1], x[:, i:], out=out[:, k:k + w]); k += w
    return out
F16 = np.empty((NPIX, NF), np.float64); t0 = time.time(); max_l4 = 0.0; sym_max = 0.0; psd_min = np.inf
for a in range(NPIX):
    Yr = Y16[ms.R[a]]; v = ms.valid[a].astype(float); c = float(ms.cnt[a]); Ap = 0.5 * (Y16 + Yr)
    B = (Ap * v[:, None]).T @ Ap / c; sym_max = max(sym_max, np.abs(B - B.T).max()); B = 0.5 * (B + B.T)
    max_l4 = max(max_l4, np.abs(B[:21, :21] - Bp4[a]).max()); F16[a] = fvec(B)
    if a % 512 == 0: psd_min = min(psd_min, float(np.linalg.eigvalsh(B).min()))
GATES['G_B16_l4_block_equals_A5_all_axes'] = bool(max_l4 < 1e-12); GATES['G_B16_symmetric_psd_sampled'] = bool(sym_max < 1e-12 and psd_min > -1e-12)
GATES['G_F16_finite_shape'] = bool(F16.shape == (3072, 40755) and F16.dtype == np.float64 and np.isfinite(F16).all())
assert GATES['G_B16_l4_block_equals_A5_all_axes'] and GATES['G_B16_symmetric_psd_sampled'] and GATES['G_F16_finite_shape'], (max_l4, sym_max, psd_min)
DIAG['F16_build'] = dict(seconds=time.time() - t0, max_l4_block_diff=max_l4, sym_max=sym_max, psd_min_sampled=psd_min, GB=F16.nbytes / 1e9)
F16_SHA64 = sha256_array(F16); F16_32 = F16.astype(np.float32); F16_SHA32 = sha256_array(F16_32)
print(f'F16 built {DIAG["F16_build"]["seconds"]:.0f}s / l4 block max diff {max_l4:.1e} / sha64 {F16_SHA64[:12]} sha32 {F16_SHA32[:12]}')

In [ ]:
# ---- 4. 正式検証電池（観測マップ・fresh 等方・hybrid・f64/f32 特徴経路）----
T1o, T2o = 39.67178834527284, 259.3375006282747
def alm_to_x(alm, M, LM):
    a = np.array([(alm[hp.Alm.getidx(LMAX_SRC, l, m)] if m >= 0 else ((-1) ** m) * np.conj(alm[hp.Alm.getidx(LMAX_SRC, l, -m)])) for (l, m) in LM]); return (a @ M.T).real
def sep_deg(a, b): return float(np.degrees(np.arccos(np.clip(np.dot(hp.pix2vec(NSIDE, a), hp.pix2vec(NSIDE, b)), -1, 1))))
def evaluate(alm, label, group):
    Sp_ref, _ = pm.scan_S(ms, hp.alm2map(hp.almxfl(alm.copy(), W16), NSIDE))          # frozen float32 pipeline on the l<=16 window
    x16 = alm_to_x(alm, M16, LM16); f64 = packed_features_into(x16[None, :], np.empty((1, NF))); S64 = (f64 @ F16.T)[0]
    f32 = packed_features_into(x16[None, :].astype(np.float32), np.empty((1, NF), np.float32)); S32 = (f32 @ F16_32.T)[0]
    a_ref, a64, a32 = int(np.argmin(Sp_ref)), int(np.argmin(S64)), int(np.argmin(S32))
    mask_ru = np.ones(NPIX, bool); mask_ru[a64] = False; ru = int(np.flatnonzero(mask_ru)[np.argmin(S64[mask_ru])])          # raw runner-up: winner only excluded
    gap = float((S64[ru] - S64[a64]) / max(abs(S64[a64]), np.finfo(float).tiny))
    anti = int(hp.vec2pix(NSIDE, *(-np.array(hp.pix2vec(NSIDE, a64))))); anti_gap = float((S64[anti] - S64[a64]) / max(abs(S64[a64]), np.finfo(float).tiny)); R_row_equal = bool(np.array_equal(ms.R[a64], ms.R[anti]))
    x4 = x16[:21]
    def T12(a): return float(x4 @ Bp4[a] @ x4), float(x4 @ Bm4[a] @ x4)
    (T1_64, T2_64), (T1_32, T2_32), (T1_r, T2_r) = T12(a64), T12(a32), T12(a_ref)
    EB = lambda t1, t2: bool(t1 <= T1o and t2 <= T2o)
    return dict(sample=label, group=group, all_axis_rel_resid_f64=float(np.abs(S64 - Sp_ref).max() / Sp_ref.max()), all_axis_rel_resid_f32=float(np.abs(S32 - Sp_ref).max() / Sp_ref.max()),
                selected_T1sel_rel_resid=float(abs(S64[a64] - Sp_ref[a_ref]) / Sp_ref[a_ref]), sel_score_rel_f32_vs_f64=float(abs(S32[a32] - S64[a64]) / max(abs(S64[a64]), np.finfo(float).tiny)),
                argmin_ref=a_ref, argmin_f64=a64, argmin_f32=a32, agree_f64=(a64 == a_ref), agree_f32=(a32 == a_ref), agree_f32_f64=(a32 == a64),
                sep_f64_deg=sep_deg(a_ref, a64), sep_f32_deg=sep_deg(a_ref, a32), sep_f32_f64_deg=sep_deg(a64, a32),
                raw_runner_up_gap=gap, runner_up_axis=ru, runner_up_sep_deg=sep_deg(a64, ru), runner_up_is_antipode=(ru == anti), antipode_axis=anti, antipode_score_gap=anti_gap, R_row_equal=R_row_equal,
                same_plane_f32=(a32 == a64 or a32 == anti), plane_folded_sep_f32_deg=min(sep_deg(a64, a32), sep_deg(anti, a32)), near_tie=(gap < 1e-4),
                T1_l24_f64=T1_64, T2_l24_f64=T2_64, T1_l24_f32=T1_32, T2_l24_f32=T2_32, T1_l24_ref=T1_r, T2_l24_ref=T2_r, EB_f64=EB(T1_64, T2_64), EB_f32=EB(T1_32, T2_32), EB_ref=EB(T1_r, T2_r))
rows = []; t0 = time.time()
cm = {'PR3_Commander': 'commander', 'PR3_NILC': 'nilc', 'PR3_SEVEM': 'sevem', 'PR3_SMICA': 'smica', 'Nofi_70GHz': 'cleaned_70GHz_v9', 'Nofi_94GHz': 'cleaned_94GHz_v9', 'Nofi_100GHz': 'cleaned_100GHz_v9', 'Nofi_143GHz': 'cleaned_143GHz_v9'}
T_SRC = hp.gauss_beam(np.radians(1.0), lmax=LMAX_SRC) * p2.pixwin_pad(128, LMAX_SRC); MAP_SHA = {}
PR4_DIR = '/content/drive/MyDrive/phase2_null/sources' if IN_COLAB else os.environ.get('A8_PR4_DIR', '')
EXPECTED_MAPS = a5p['maps']                                         # 10 maps with frozen SHA (A5 provenance)
paths = {k: os.path.join(CA, 'data', 'real', f'map_{v}_nside_128.fits') for k, v in cm.items()}
paths.update({'PR4_Sevem': os.path.join(PR4_DIR, 'npipe_sevem_128.fits'), 'PR4_Commander': os.path.join(PR4_DIR, 'npipe_commander_128.fits')})
present = {k: p for k, p in paths.items() if os.path.exists(p)}
for k, p in present.items(): MAP_SHA[k] = sha256_file(p)
GATES['G_observed_map_sha'] = all(MAP_SHA.get(k) == v for k, v in EXPECTED_MAPS.items() if k in MAP_SHA)
GATES['G_observed_inventory_10'] = (set(MAP_SHA) == set(EXPECTED_MAPS)) if A8_MODE == 'official' else (set(cm) <= set(MAP_SHA))
assert GATES['G_observed_map_sha'] and GATES['G_observed_inventory_10'], (set(EXPECTED_MAPS) - set(MAP_SHA), [k for k in MAP_SHA if MAP_SHA[k] != EXPECTED_MAPS.get(k)])
for k, p in present.items():
    rows.append(evaluate(hp.almxfl(hp.map2alm(hp.read_map(p), lmax=LMAX_SRC), 1.0 / np.maximum(T_SRC, 1e-12)), k, 'observed'))
N_ISO = 1000 if A8_MODE == 'official' else 40; N_HYB = 200 if A8_MODE == 'official' else 20
seeds = np.random.default_rng(np.random.SeedSequence([20260908, 8, 1])).integers(2**31 - 1, size=N_ISO); DIAG['validation_seeds'] = dict(recipe='SeedSequence([20260908,8,1]) -> integers(2^31-1, N_ISO); hybrid SeedSequence([20260908,8,2])', seed_array_sha256=sha256_array(seeds))
for i in range(N_ISO):
    np.random.seed(int(seeds[i])); rows.append(evaluate(hp.synalm(CL, lmax=LMAX_SRC), f'iso_{i}', 'fresh_isotropic'))
# hybrid: topology l2-4 (A11 frozen E7 covariance) + isotropic l5-16
A11 = os.path.join(MT, 'results', 'step1_phaseA', 'A11_freeze'); A11C = os.path.join(A11, 'official', 'cov_cache')
GATES['G_a11_freeze_manifest_sha'] = (sha256_file(os.path.join(A11, 'freeze_manifest.json')) == '6000d7b7049dc5232f3e87daab2d4e56cb069cd15145f252ba4a548bf94df981')
GATES['G_a11_official_provenance_sha'] = (sha256_file(os.path.join(A11, 'official', 'a11_provenance.json')) == '1cdb719c9a914172a8ce5d19bb43cea91f88960acdd77ca1fd96ef57bfff2656')
a11p = json.load(open(os.path.join(A11, 'official', 'a11_provenance.json')))
GATES['G_a11_official_status'] = (a11p['status'] == 'OFFICIAL' and a11p['OFFICIAL'] is True and a11p['gate_inventory_exact'] is True and all(a11p['gates'][k] is True for k in a11p['required_gates']))
target = next(c for c in a11p['cases'] if c['case'] == 'E7_b1_A')
cov_tags = [k for k, v in a11p['covariance_intake_meta'].items() if v['cache']['manifest']['topology'] == 'E7' and v['cache']['manifest']['params'] == target['shape'] and v['cache']['manifest']['x0'] == target['base']]
GATES['G_a11_cov_tag_unique'] = (len(cov_tags) == 1); assert GATES['G_a11_cov_tag_unique'], cov_tags
covf = os.path.join(A11C, f'cov_{cov_tags[0][:20]}.npy'); GATES['G_a11_cov_sha'] = (sha256_file(covf) == a11p['covariance_intake_meta'][cov_tags[0]]['cache']['cov_file_sha256'])
assert GATES['G_a11_freeze_manifest_sha'] and GATES['G_a11_official_provenance_sha'] and GATES['G_a11_official_status'] and GATES['G_a11_cov_sha']
Csym, C4, meta = t1.load_cov_full(covf, 4)                              # (complex full-m symmetrised covariance, real-basis covariance, meta)
GATES['G_a11_cov_basis_matches_M4'] = bool(np.abs((M4 @ Csym @ M4.conj().T).real - C4).max() < 1e-12 * np.abs(C4).max())   # real basis of the hybrid draw == A8a M4 (frozen bridge)
assert GATES['G_a11_cov_basis_matches_M4']
w, V = np.linalg.eigh(C4); C4h = V @ np.diag(np.sqrt(np.maximum(w, 0))) @ V.T
rng_h = np.random.default_rng(np.random.SeedSequence([20260908, 8, 2]))
for i in range(N_HYB):
    x4 = rng_h.standard_normal(21) @ C4h.T; a4 = M4.conj().T @ x4                     # topology l2-4 coefficients (frozen basis, complex)
    np.random.seed(int(rng_h.integers(2**31 - 1))); alm = hp.synalm(CL, lmax=LMAX_SRC)  # isotropic all l, then overwrite l<=4 with the topology draw
    for (l, m) in LM4:
        if m >= 0: alm[hp.Alm.getidx(LMAX_SRC, l, m)] = a4[LM4.index((l, m))]
    rows.append(evaluate(alm, f'hyb_{i}', 'hybrid_E7_l24_iso_l516'))
val = pd.DataFrame(rows); print(f'validation rows {len(val)} ({time.time()-t0:.0f}s)')
val['margin_bin'] = pd.cut(val.raw_runner_up_gap, [0, 1e-6, 1e-4, 1e-2, 1e-1, np.inf], labels=['<1e-6', '1e-6..1e-4', '1e-4..1e-2', '1e-2..1e-1', '>1e-1'], include_lowest=True)
summary = dict(n=len(val), max_all_axis_rel_resid_f64=float(val.all_axis_rel_resid_f64.max()), max_all_axis_rel_resid_f32=float(val.all_axis_rel_resid_f32.max()),
               argmin_agree_f64=float(val.agree_f64.mean()), argmin_agree_f32=float(val.agree_f32.mean()), argmin_agree_f32_f64=float(val.agree_f32_f64.mean()),
               EB_mismatch_f64=int((val.EB_f64 != val.EB_ref).sum()), near_tie_count=int(val.near_tie.sum()),
               f32_f64_agree_by_margin={str(k): float(v) for k, v in val.groupby('margin_bin', observed=True).agree_f32_f64.mean().items()},
               per_group_argmin_agree_f64={k: float(v) for k, v in val.groupby('group').agree_f64.mean().items()})
summary['EB_mismatch_f32'] = int((val.EB_f32 != val.EB_ref).sum())
GATES['G_F16_pipeline_battery_f64'] = bool(summary['max_all_axis_rel_resid_f64'] < 1e-5 and summary['argmin_agree_f64'] == 1.0 and summary['EB_mismatch_f64'] == 0)
GATES['G_F16_observed_maps_agree'] = bool(val[val.group == 'observed'].agree_f64.all() and val[val.group == 'observed'].agree_f32.all())
# float32 formal eligibility (v1.1.1): non-near-tie -> exact argmin agreement; near-tie with axis change -> selection score (1e-4), T1/T2 (1e-6 rel) and Event B must agree
summary['antipode_exact_tie_frac'] = float((val.antipode_score_gap.abs() <= 1e-12).mean()); summary['runner_up_is_antipode_frac'] = float(val.runner_up_is_antipode.mean()); summary['R_row_equal_frac'] = float(val.R_row_equal.mean())
nt = val[val.near_tie]; nnt = val[~val.near_tie]
f32_score_ok = bool((val.all_axis_rel_resid_f32 < 1e-5).all())
def nt_row_ok(r): return bool(r.agree_f32_f64 or (r.sel_score_rel_f32_vs_f64 < 1e-4 and abs(r.T1_l24_f32 - r.T1_l24_f64) <= 1e-6 * max(abs(r.T1_l24_f64), 1e-300) and abs(r.T2_l24_f32 - r.T2_l24_f64) <= 1e-6 * max(abs(r.T2_l24_f64), 1e-300) and r.EB_f32 == r.EB_f64))
def axis_row_ok(r): return bool(r.agree_f32_f64 or r.same_plane_f32 or r.plane_folded_sep_f32_deg <= 4.0)   # axis-output eligibility: same physical plane or grid-neighbour
nt_ok = bool(all(nt_row_ok(r) for r in nt.itertuples())) if len(nt) else True
GATES['G_F16_pipeline_battery_f32_non_tie'] = bool(nnt.agree_f32.all() and f32_score_ok)
GATES['G_F16_f32_eventB'] = bool((val.EB_f32 == val.EB_ref).all())
mism = val[~val.agree_f32_f64]
FLAGS = dict(F_F16_f32_near_tie_count=int(len(nt)), F_F16_f32_near_tie_axis_mismatch=int((~nt.agree_f32).sum()) if len(nt) else 0, F_F16_f32_near_tie_rule_ok=nt_ok,
             F_F16_f32_axis_mismatch_count=int(len(mism)), F_F16_f32_axis_mismatch_max_plane_folded_sep_deg=(float(mism.plane_folded_sep_f32_deg.max()) if len(mism) else 0.0),
             float32_eligible_for_event_outputs=bool(GATES['G_F16_pipeline_battery_f32_non_tie'] and GATES['G_F16_f32_eventB'] and nt_ok),
             float32_eligible_for_axis_outputs=bool(GATES['G_F16_pipeline_battery_f32_non_tie'] and all(axis_row_ok(r) for r in mism.itertuples())))
FLAGS['float32_eligible_for_rules'] = FLAGS['float32_eligible_for_event_outputs']     # event-output eligibility (rules must interpret axis outputs with float32_eligible_for_axis_outputs)
# near-tie battery: per-bin counts/agreement, and the K smallest-margin fresh samples (smallest-margin diagnostic)
bins = val.groupby('margin_bin', observed=False).agg(n=('sample', 'size'), agree_f32_f64=('agree_f32_f64', 'mean'), max_sep_f32_deg=('sep_f32_deg', 'max')).reset_index()
K = 20; smallest = val[val.group != 'observed'].nsmallest(K, 'raw_runner_up_gap')
summary.update(near_tie_bins=bins.astype(str).to_dict(orient='records'), smallest_margin_K=dict(K=K, agree_f32_f64=float(smallest.agree_f32_f64.mean()), max_sep_f32_deg=float(smallest.sep_f32_deg.max()), min_gap=float(smallest.raw_runner_up_gap.min())))
DIAG['validation'] = summary; DIAG['flags'] = FLAGS; print(json.dumps(dict(summary=summary, flags=FLAGS), indent=1)[:1500])

In [ ]:
# ---- 5. atomic 保存 → fresh reopen 再検査 → status ----
def atomic_write_json(obj, path):
    tmp = path + '.tmp'
    with open(tmp, 'w') as fh: json.dump(obj, fh, indent=1); fh.flush(); os.fsync(fh.fileno())
    os.replace(tmp, path)
def atomic_save_npy(arr, path):
    tmp = path + '.tmp.npy'
    with open(tmp, 'wb') as fh: np.save(fh, arr); fh.flush(); os.fsync(fh.fileno())
    os.replace(tmp, path)
F64_PATH = os.path.join(OUT, 's1_Bplus_featurestack_l2_16_N16_common_v1_float64.npy'); F32_PATH = os.path.join(OUT, 's1_Bplus_featurestack_l2_16_N16_common_v1_float32.npy')
atomic_save_npy(F16, F64_PATH); atomic_save_npy(F16_32, F32_PATH)
MANIFEST = dict(schema='featurestack_manifest_v1', artifact='S+ feature stack, l=2..16, N16 common mask, 3072 axes', shape=[3072, 40755], dtype_primary='float64',
                packing='upper-triangular (i<=j) row-major; off-diagonal entries x2 so that S+(x,a) = f(x).F[a] with f = x_i x_j (i<=j)', axis_pixel_ids='0..3071 RING (nside 16)',
                basis_lm=[[int(l), int(m), cs] for (l, m, cs) in RB16], basis_convention='t2b2_bridge M_matrix(range(2,17)): m=0 Re; cos sqrt2 Re; sin sqrt2 Im; frozen l2-4 ordering is the prefix',
                window='FL(planck, nside16) x [l=2..16]', normalization='S = mean over valid pairs of ((T+T[R])/2)^2, / cnt (frozen pm.scan_S)',
                packing_schema_version='packed_upper_v1',
                sha256=dict(F16_float64_array=F16_SHA64, F16_float32_array=F16_SHA32, F16_float64_file=sha256_file(F64_PATH), F16_float32_file=sha256_file(F32_PATH),
                            iu0=sha256_array(iu[0].astype(np.int32)), iu1=sha256_array(iu[1].astype(np.int32)), basis_lm=hashlib.sha256(json.dumps([[int(l), int(m), cs] for (l, m, cs) in RB16]).encode()).hexdigest(),
                            axis_pixel_ids=sha256_array(np.arange(3072, dtype=np.int32))),
                inputs=dict(**FROZEN, **INPUT_SHA), frozen_repos=REPOS, build=DIAG['F16_build'], validation=DIAG['validation'], notebook=DIAG['notebook'], env_fingerprint=ENV_FINGERPRINT)
# fresh reopen (mmap) re-verification
Fm = np.load(F64_PATH, mmap_mode='r'); Fm32 = np.load(F32_PATH, mmap_mode='r')
GATES['G_artifact_reopen'] = bool(Fm.shape == (3072, 40755) and Fm.dtype == np.float64 and Fm32.shape == (3072, 40755) and Fm32.dtype == np.float32
                                  and sha256_array(np.asarray(Fm)) == F16_SHA64 and sha256_array(np.asarray(Fm32)) == F16_SHA32)
xchk = np.random.default_rng(7).standard_normal(285)
# explicit packing check: f(x).F[a] must equal x^T B_a x for a reconstructed B_a
a_chk = 1134; Yr = Y16[ms.R[a_chk]]; v = ms.valid[a_chk].astype(float); Ap = 0.5 * (Y16 + Yr); B_chk = (Ap * v[:, None]).T @ Ap / float(ms.cnt[a_chk]); B_chk = 0.5 * (B_chk + B_chk.T)
_q = xchk @ B_chk @ xchk; GATES['G_artifact_packing_roundtrip'] = bool(abs(packed_features_into(xchk[None, :], np.empty((1, NF)))[0] @ np.asarray(Fm[a_chk]) - _q) < max(1e-9 * abs(_q), 1e-12))
_tmp = os.path.join(OUT, 'a8a_validation.csv.tmp'); val.to_csv(_tmp, index=False); os.replace(_tmp, os.path.join(OUT, 'a8a_validation.csv'))
MANIFEST_PATH = F64_PATH.replace('_float64.npy', '_manifest.json'); atomic_write_json(MANIFEST, MANIFEST_PATH); MANIFEST_SHA = sha256_file(MANIFEST_PATH)
GATES['G_manifest_roundtrip'] = (json.load(open(MANIFEST_PATH)) == json.loads(json.dumps(MANIFEST)))
REQ_COMMON = ['G_pem_commit', 'G_pem_origin', 'G_pem_clean_preimport', 'G_cmbanom_commit', 'G_cmbanom_origin', 'G_cmbanom_clean_preimport', 'G_mt_commit', 'G_mt_origin', 'G_mt_clean_preimport',
              'G_a5_manifest_sha', 'G_a5_provenance_in_manifest', 'G_a5_bstack_file_sha', 'G_pem_src_sha', 'G_mt_src_sha', 'G_live_dependency_versions', 'G_live_import_paths', 'G_live_module_sha',
              'G_pipeline_objects', 'G_mask_R_valid_cnt_sha', 'G_a5_input_manifest_sha', 'G_FL_matches_A5', 'G_CL_matches_A5', 'G_W4_matches_A5', 'G_W16_matches_A5', 'G_basis_extension', 'G_Y16_prefix_equals_Y4',
              'G_a5_bstack_array_sha', 'G_B16_l4_block_equals_A5_all_axes', 'G_B16_symmetric_psd_sampled', 'G_F16_finite_shape', 'G_a11_freeze_manifest_sha', 'G_a11_official_provenance_sha', 'G_a11_official_status',
              'G_a11_cov_tag_unique', 'G_a11_cov_sha', 'G_a11_cov_basis_matches_M4', 'G_observed_map_sha', 'G_observed_inventory_10', 'G_F16_pipeline_battery_f64', 'G_F16_observed_maps_agree', 'G_F16_pipeline_battery_f32_non_tie', 'G_F16_f32_eventB',
              'G_artifact_reopen', 'G_artifact_packing_roundtrip', 'G_manifest_roundtrip']
REQUIRED = REQ_COMMON + (['G_colab_runtime', 'G_env_lock', 'G_smoke_provenance_chain', 'G_notebook_live_source'] if A8_MODE == 'official' else ['G_env_lock_candidate_complete', 'G_notebook_head_available'])
GATES = {k: bool(v) for k, v in GATES.items()}; EXACT = (set(GATES) == set(REQUIRED)); RUN_PASS = EXACT and all(GATES[k] for k in REQUIRED)
STATUS = ('FEATURESTACK_VALID' if RUN_PASS and A8_MODE == 'official' else 'SMOKE_PASS' if RUN_PASS else 'FAILED')
prov = dict(notebook=f'Step1 Phase A-8a feature stack v1.1.2 [{A8_MODE}]', status=STATUS, FEATURESTACK_VALID=(STATUS == 'FEATURESTACK_VALID'), timestamp=datetime.datetime.now(datetime.timezone.utc).isoformat(),
            gates=GATES, flags=FLAGS, required_gates=REQUIRED, gate_inventory_exact=EXACT, manifest=MANIFEST, manifest_file=os.path.basename(MANIFEST_PATH), manifest_file_sha256=MANIFEST_SHA, map_sha256=MAP_SHA,
            environment=dict(python=PYV, live=LIVE, platform=PLATFORM, blas=BLAS, fingerprint=ENV_FINGERPRINT, lock_sha256=LOCK_SHA), notebook_identity=DIAG['notebook'], diagnostics=DIAG,
            git_calls=GIT_LOG, outputs=dict(validation_csv_sha256=sha256_file(os.path.join(OUT, 'a8a_validation.csv'))))
atomic_write_json(prov, os.path.join(OUT, 'a8a_provenance.json'))
_re = json.load(open(os.path.join(OUT, 'a8a_provenance.json'))); assert _re['manifest_file_sha256'] == MANIFEST_SHA and _re['manifest'] == json.loads(json.dumps(MANIFEST)), 'provenance reopen binding failed'
if STATUS == 'SMOKE_PASS':
    LOCK_CANDIDATE.update(smoke_status='SMOKE_PASS', smoke_provenance_path=os.path.join(OUT, 'a8a_provenance.json'), smoke_provenance_sha256=sha256_file(os.path.join(OUT, 'a8a_provenance.json')), canonical_notebook_head_sha=NB_HEAD_SHA, generated=datetime.datetime.now(datetime.timezone.utc).isoformat())
    atomic_write_json(LOCK_CANDIDATE, LOCK_PATH); print('[smoke] environment lock written')
print('STATUS =', STATUS, '/ saved:', OUT); failed = {k: v for k, v in GATES.items() if not v}
assert RUN_PASS, f'A8a FAILED: failed={failed} missing={set(REQUIRED)-set(GATES)} unregistered={set(GATES)-set(REQUIRED)}'

## 実行手順
1. notebook をリポジトリ直下に commit・push。
2. **smoke**：コピーの先頭に `A8_MODE = 'smoke'` を追加して実行（約 10 分・env lock 生成）。
3. **official**：純正版を Runtime restart → Run all（約 30–40 分：観測 8–10・等方 1000・hybrid 200 の検証電池）。
4. `a8a_v1.1.2_official/` の `a8a_provenance.json`・`a8a_validation.csv`・`*_manifest.json` と全セル出力を返送（`.npy` 1.0 GB＋0.5 GB は Drive に置いたまま）。
A8b（ベンチマーク）は A8a の FEATURESTACK_VALID と `float32_eligible_for_rules` を入力に fresh runtime で実行する。